# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhassanbuilds/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

if token:
    print("✅ HF_TOKEN loaded successfully!")
else:
    print("❌ HF_TOKEN not found.")

✅ HF_TOKEN loaded successfully!


In [3]:
%pip -q install duckdb huggingface_hub

In [4]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("✅ Connected to FlyRank Internship Warehouse")

✅ Connected to FlyRank Internship Warehouse


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## Unit of Analysis + Time Window

**Unit of Analysis:** One row represents the daily search performance of one content page for one client.

**Time Window:** For this assignment, I use data from **March 2026** (2026-03-01 to 2026-03-31). This mid-panel month provides enough historical information while avoiding the final month of the dataset, which is better reserved for future evaluation.

The project aims to rank content pages that should be prioritized for review and refresh based on their historical search performance and engagement signals.

In [22]:
con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows
0,2025-01-27,2026-06-30,78835655


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Fields

### Features

- gsc_impressions
- gsc_clicks
- gsc_avg_position
- active_days (derived from the number of available daily records)

These features describe the historical search performance of each content page and are available before making a refresh decision.

### Label / Proxy
The project uses a **refresh priority recommendation** as a proxy rather than a direct label. Since there is no field that explicitly states whether a page needs refreshing, the recommendation is based on observed search performance patterns.

### Context
- report_date
- client_hash_id
- content_hash_id

These fields identify the page, client, and time period but are not used directly as predictive features.

### Excluded
Google Analytics metrics and AI traffic fields are excluded from the initial model because the Refresh lane primarily focuses on search performance. These fields may be explored in later iterations but are not required for the first version of the model.

In [23]:
con.sql(f"""
SELECT
    COUNT(DISTINCT content_hash_id) AS content_pages,
    COUNT(DISTINCT client_hash_id) AS clients
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_pages,clients
0,427292,70


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verification

The following queries verify the assumptions used in this data contract:

- Verify the warehouse date range.
- Verify the number of unique content pages and clients.
- Verify that Google Search Console data is available for the selected month using `gsc_data_available IS TRUE`.
- Build a small feature frame from the verified data for the Refresh lane.

In [24]:
con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM {TABLES['fact_daily']}
WHERE gsc_data_available IS TRUE
AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


In [25]:
feature_df = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position,
    COUNT(*) AS active_days
FROM {TABLES['fact_daily']}
WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
AND gsc_data_available IS TRUE
GROUP BY content_hash_id
LIMIT 10
""").df()

feature_df

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,avg_position,active_days
0,content_320f1ceaf171c5a9,354.0,1.0,9.809343,31
1,content_2ca7362a4e21cbea,459.0,1.0,7.425070,30
2,content_f2e4f70271098e06,1294.0,2.0,5.766722,31
3,content_a80297a7efbc93d9,105.0,1.0,23.621154,26
4,content_55b8c3baeccccdec,130.0,0.0,23.229916,31
5,content_1130543e8b6717de,424.0,0.0,6.904439,31
6,content_850ce72b065c1848,1506.0,3.0,5.435268,31
7,content_58917406ff06d022,4519.0,2.0,3.423326,31
8,content_acac9f7fbf9b5e54,16.0,0.0,34.972222,12
9,content_369f1dd57957e310,7307.0,6.0,5.418526,31


## Five Features

| Feature | Available when? |
|----------|-----------------|
| gsc_impressions | Available before making the refresh decision because it comes from historical Search Console data. |
| gsc_clicks | Available before the decision because it summarizes previous user interactions. |
| gsc_avg_position | Available before the decision because it reflects historical search ranking performance. |
| active_days | Available before the decision because it counts days with recorded data during the selected month. |
| content_hash_id | Used to uniquely identify each content page throughout the analysis. |

In [26]:
feature_df = feature_df.copy()

# Deliberately create a simple label-derived column (for demonstration only)
feature_df["high_impressions"] = (
    feature_df["impressions"] > feature_df["impressions"].median()
).astype(int)

feature_df.head()

,content_hash_id,impressions,clicks,avg_position,active_days,high_impressions
0,content_320f1ceaf171c5a9,354.0,1.0,9.809343,31,0
1,content_2ca7362a4e21cbea,459.0,1.0,7.425070,30,1
2,content_f2e4f70271098e06,1294.0,2.0,5.766722,31,1
3,content_a80297a7efbc93d9,105.0,1.0,23.621154,26,0
4,content_55b8c3baeccccdec,130.0,0.0,23.229916,31,0


### Leakage Demonstration

The `high_impressions` column is intentionally created from the target-related information to demonstrate what label leakage looks like.

If a model were trained using this column, performance would appear unrealistically high because the feature already contains information about the outcome.

This column is only included for demonstration and would be removed before building a real machine learning model.

In [27]:
# Remove the leakage column
feature_df = feature_df.drop(columns=["high_impressions"])

feature_df.head()

,content_hash_id,impressions,clicks,avg_position,active_days
0,content_320f1ceaf171c5a9,354.0,1.0,9.809343,31
1,content_2ca7362a4e21cbea,459.0,1.0,7.425070,30
2,content_f2e4f70271098e06,1294.0,2.0,5.766722,31
3,content_a80297a7efbc93d9,105.0,1.0,23.621154,26
4,content_55b8c3baeccccdec,130.0,0.0,23.229916,31


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limits

This dataset supports decision-support analysis based on historical search performance, but it has several limitations. The panel is unbalanced because different clients have different amounts of historical data available. Some rows contain only Google Search Console information while others also include Google Analytics metrics. The data can identify patterns associated with content performance, but it cannot explain all factors that influence search performance, such as competitor actions, content quality, or search engine algorithm updates. Therefore, the recommendations produced by this project should support human decision-making rather than replace it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.